In [1]:
from datetime import datetime, timedelta, timezone
from dateutil.rrule import rrule, DAILY, WEEKLY, MONTHLY, FR

import polars as pl

from backtester import samplers

In [2]:
t0 = datetime(2025, 1, 1, tzinfo=timezone.utc)
tf = datetime(2025, 12, 31, tzinfo=timezone.utc)
dt = timedelta(hours=1)

In [3]:
#     For now, hardcode the following contract introduction rules:
#         - seven dailies (08:00 UTC)
#         - four weeklies (Fridays at 08:00 UTC)
#         - three monthlies (last Friday of each month at 08:00 UTC)
#         - four quarterlies (last Friday of Mar, Jun, Sep, Dec at 08:00 UTC)
#         - strikes from 10% to 190% moneyness in 10% increments
#         - on duplicates, keep longest tenor
daily = rrule(DAILY, byhour=8)
weekly = rrule(WEEKLY, byweekday=FR, byhour=8)
monthly = rrule(MONTHLY, byweekday=FR(-1), byhour=8)
quarterly = rrule(MONTHLY, bymonth=(3, 6, 9, 12), byweekday=FR(-1), byhour=8)

In [4]:
def _get_option_listings_and_expiries(
    t0: datetime,
    tf: datetime,
    rule: rrule,
) -> tuple[list[datetime], list[datetime]]:
    r = rule.replace(dtstart=t0)

    prev_ = r.before(t0, inc=True)  # expiry at or before t0
    inside = list(r.between(t0, tf, inc=True))  # expiries in [t0, tf]
    next_ = r.after(tf, inc=True)  # expiry at or after tf

    out = []
    if prev_ is not None:
        out.append(prev_)
    out.extend(inside)
    if next_ is not None and next_ != out[-1]:
        out.append(next_)

    expiries = out

    listings = []
    for expiry in expiries:
        listings.append(
            r.replace(dtstart=datetime(1, 1, 1, tzinfo=timezone.utc)).before(
                expiry, inc=False
            )
        )

    return (listings, expiries)

In [5]:
def _get_option_listings_and_expiries(
    t0: datetime,
    tf: datetime,
    rule: rrule,
) -> pl.DataFrame:
    r = rule.replace(dtstart=datetime(1, 1, 1, tzinfo=timezone.utc))
    prev_: datetime = r.before(t0, inc=True)  # expiry at or before t0
    next_: datetime = r.after(tf, inc=True)  # expiry at or after tf
    inside: list[datetime] = r.between(t0, tf, inc=True)  # expiries in [t0, tf]
    return pl.DataFrame({"listing": [prev_] + inside, "expiry": inside + [next_]})

In [6]:
listings_and_expiries = _get_option_listings_and_expiries(t0, tf, monthly)
listings_and_expiries

listing,expiry
"datetime[μs, UTC]","datetime[μs, UTC]"
2024-12-27 08:00:00 UTC,2025-01-31 08:00:00 UTC
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC
2025-02-28 08:00:00 UTC,2025-03-28 08:00:00 UTC
2025-03-28 08:00:00 UTC,2025-04-25 08:00:00 UTC
2025-04-25 08:00:00 UTC,2025-05-30 08:00:00 UTC
…,…
2025-08-29 08:00:00 UTC,2025-09-26 08:00:00 UTC
2025-09-26 08:00:00 UTC,2025-10-31 08:00:00 UTC
2025-10-31 08:00:00 UTC,2025-11-28 08:00:00 UTC


In [7]:
paths_mark = samplers.get_paths_mark(t0, tf, dt)
paths_mark.collect()

time_start,time_end,name,price
"datetime[μs, UTC]","datetime[μs, UTC]",str,f64
2025-01-01 00:00:00 UTC,2025-01-01 01:00:00 UTC,"""asset_0""",0.992719
2025-01-01 01:00:00 UTC,2025-01-01 02:00:00 UTC,"""asset_0""",0.98752
2025-01-01 02:00:00 UTC,2025-01-01 03:00:00 UTC,"""asset_0""",0.969519
2025-01-01 03:00:00 UTC,2025-01-01 04:00:00 UTC,"""asset_0""",0.976953
2025-01-01 04:00:00 UTC,2025-01-01 05:00:00 UTC,"""asset_0""",0.971486
…,…,…,…
2025-12-30 19:00:00 UTC,2025-12-30 20:00:00 UTC,"""asset_0""",0.482819
2025-12-30 20:00:00 UTC,2025-12-30 21:00:00 UTC,"""asset_0""",0.478433
2025-12-30 21:00:00 UTC,2025-12-30 22:00:00 UTC,"""asset_0""",0.474061


In [8]:
# Create a moneynesses to be cross-joined with listing/price.
moneynesses = pl.DataFrame({"moneyness": [0.1 * i for i in range(1, 20)]})
moneynesses

moneyness
f64
0.1
0.2
0.3
0.4
0.5
…
1.5
1.6
1.7


In [9]:
# Join so we have listing and spot price at listing time for each expiry.
left = listings_and_expiries.lazy()
right = paths_mark

joined = left.join_asof(right, left_on="listing", right_on="time_end").drop_nulls()
joined.collect()

listing,expiry,time_start,time_end,name,price
"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,f64
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,2025-01-31 07:00:00 UTC,2025-01-31 08:00:00 UTC,"""asset_0""",0.521391
2025-02-28 08:00:00 UTC,2025-03-28 08:00:00 UTC,2025-02-28 07:00:00 UTC,2025-02-28 08:00:00 UTC,"""asset_0""",0.700255
2025-03-28 08:00:00 UTC,2025-04-25 08:00:00 UTC,2025-03-28 07:00:00 UTC,2025-03-28 08:00:00 UTC,"""asset_0""",0.832082
2025-04-25 08:00:00 UTC,2025-05-30 08:00:00 UTC,2025-04-25 07:00:00 UTC,2025-04-25 08:00:00 UTC,"""asset_0""",0.504437
2025-05-30 08:00:00 UTC,2025-06-27 08:00:00 UTC,2025-05-30 07:00:00 UTC,2025-05-30 08:00:00 UTC,"""asset_0""",0.563592
…,…,…,…,…,…
2025-08-29 08:00:00 UTC,2025-09-26 08:00:00 UTC,2025-08-29 07:00:00 UTC,2025-08-29 08:00:00 UTC,"""asset_0""",0.494363
2025-09-26 08:00:00 UTC,2025-10-31 08:00:00 UTC,2025-09-26 07:00:00 UTC,2025-09-26 08:00:00 UTC,"""asset_0""",0.678821
2025-10-31 08:00:00 UTC,2025-11-28 08:00:00 UTC,2025-10-31 07:00:00 UTC,2025-10-31 08:00:00 UTC,"""asset_0""",0.580743


In [10]:
# Polars cross-join.
struck = joined.join(moneynesses.lazy(), how="cross").with_columns(
    [(pl.col("moneyness") * pl.col("price")).round(2).alias("strike")]
)
struck.collect()

listing,expiry,time_start,time_end,name,price,moneyness,strike
"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,2025-01-31 07:00:00 UTC,2025-01-31 08:00:00 UTC,"""asset_0""",0.521391,0.1,0.05
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,2025-01-31 07:00:00 UTC,2025-01-31 08:00:00 UTC,"""asset_0""",0.521391,0.2,0.1
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,2025-01-31 07:00:00 UTC,2025-01-31 08:00:00 UTC,"""asset_0""",0.521391,0.3,0.16
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,2025-01-31 07:00:00 UTC,2025-01-31 08:00:00 UTC,"""asset_0""",0.521391,0.4,0.21
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,2025-01-31 07:00:00 UTC,2025-01-31 08:00:00 UTC,"""asset_0""",0.521391,0.5,0.26
…,…,…,…,…,…,…,…
2025-12-26 08:00:00 UTC,2026-01-30 08:00:00 UTC,2025-12-26 07:00:00 UTC,2025-12-26 08:00:00 UTC,"""asset_0""",0.447245,1.5,0.67
2025-12-26 08:00:00 UTC,2026-01-30 08:00:00 UTC,2025-12-26 07:00:00 UTC,2025-12-26 08:00:00 UTC,"""asset_0""",0.447245,1.6,0.72
2025-12-26 08:00:00 UTC,2026-01-30 08:00:00 UTC,2025-12-26 07:00:00 UTC,2025-12-26 08:00:00 UTC,"""asset_0""",0.447245,1.7,0.76


In [11]:
struck.select(["listing", "expiry", "strike"]).collect()  # just the relevant columns

listing,expiry,strike
"datetime[μs, UTC]","datetime[μs, UTC]",f64
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,0.05
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,0.1
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,0.16
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,0.21
2025-01-31 08:00:00 UTC,2025-02-28 08:00:00 UTC,0.26
…,…,…
2025-12-26 08:00:00 UTC,2026-01-30 08:00:00 UTC,0.67
2025-12-26 08:00:00 UTC,2026-01-30 08:00:00 UTC,0.72
2025-12-26 08:00:00 UTC,2026-01-30 08:00:00 UTC,0.76
